# Polaris Purge Practice — Catalog / Table / View

**Purpose:** Practice drop-with-purge on local OrbStack cluster  
**Observe:** MinIO before/after each purge via http://192.168.139.2:9001  

## What we do
```
Setup:    cleanup any leftover resources → create fresh catalog/ns/table/view
Part 1b:  write data to table (pyarrow + Iceberg REST commit)
Part 2:   observe MinIO before purge
Part 3:   view purge  → no MinIO change
Part 4:   table purge → Parquet files removed
Part 5:   catalog purge → base location removed
Part 6:   verify everything gone
```


In [1]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import sys
from polaris_test_utils import *
require_not_prod("purge practice")  # PROD-safe: refuses to run against company PROD

tok = root_token()
print(f"✅ Root token acquired")
print(f"Polaris: {POLARIS_URL}")
print(f"MinIO:   http://192.168.139.2:9001")

# Practice resource names
PRACTICE_CATALOG  = "purge-practice"
PRACTICE_NS       = "practice-ns"
PRACTICE_TABLE    = "practice-table"
PRACTICE_VIEW     = "practice-view"
PRACTICE_LOCATION = "s3a://data-catalog-bucket/purge-practice/"

# MinIO credentials (direct — STS not used)
MINIO_ENDPOINT   = "http://192.168.139.2:9000"
MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "bhEenDzTEAVPKbukhuY0tRY4WjpZXzh0"


🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
✅ Root token acquired
Polaris: http://192.168.139.2:8181
MinIO:   http://192.168.139.2:9001


## Cleanup — remove any leftover resources from previous runs

In [2]:
# Cleanup: delete practice catalog (and everything inside it) if exists
print("Cleaning up leftover resources...")

def must_delete(url, label, allow_status=()):
    """Delete and raise on unexpected status. allow_status = extra OK codes."""
    r = requests.delete(url, headers=h(tok))
    if r.status_code in (200, 204) or r.status_code in allow_status:
        print(f"  ✅ deleted {label} [{r.status_code}]")
    elif r.status_code == 404:
        print(f"  ⏭️  not found (skip): {label}")
    else:
        msg = ""
        try:
            msg = r.json().get("error", {}).get("message", r.text[:200])
        except Exception:
            msg = r.text[:200]
        raise RuntimeError(f"Cleanup failed [{r.status_code}] {label}: {msg}")

catalogs = requests.get(f"{BASE_MGMT}/catalogs", headers=h(tok)).json().get("catalogs", [])
if any(c["name"] == PRACTICE_CATALOG for c in catalogs):
    # ENABLE drop-with-purge so stuck entities in the async cleanup/trash queue
    # (e.g. polaris-cleanup-trash-ns) can be fully purged.
    rg = requests.get(f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}", headers=h(tok))
    cat = rg.json()
    ver = cat["entityVersion"]
    merged = {**cat.get("properties", {}),
              "polaris.config.drop-with-purge.enabled": "true",
              "polaris.config.purge-view-metadata-on-drop": "false"}
    requests.put(f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}", headers=h(tok), json={
        "currentEntityVersion": ver, "properties": merged
    })
    print("  enabled drop-with-purge for cleanup")
    requests.put(f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}/catalog-roles/catalog_admin/grants",
                 headers=h(tok),
                 json={"grant": {"type": "catalog", "privilege": "CATALOG_MANAGE_CONTENT"}})
    print("  granted CATALOG_MANAGE_CONTENT for purge")


    ns_r = requests.get(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces", headers=h(tok))
    for ns in ns_r.json().get("namespaces", []):
        ns_name = ns[0] if isinstance(ns, list) else ns

        # Tables (allow purge)
        t_r = requests.get(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{ns_name}/tables", headers=h(tok))
        for t in t_r.json().get("identifiers", []):
            must_delete(
                f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{ns_name}/tables/{t['name']}?purgeRequested=true",
                f"table: {ns_name}.{t['name']}"
            )

        # Views — try with purge flag (handles stuck trash-queue entities)
        v_r = requests.get(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{ns_name}/views", headers=h(tok))
        for v in v_r.json().get("identifiers", []):
            must_delete(
                f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{ns_name}/views/{v['name']}?purgeRequested=true",
                f"view: {ns_name}.{v['name']}",
                allow_status=(403,)  # stuck trash-queue views handled by next cell
            )

        # Namespace
        must_delete(
            f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{ns_name}",
            f"namespace: {ns_name}"
        )

    # Catalog (with purge)
    must_delete(
        f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}?purgeRequested=true",
        f"catalog: {PRACTICE_CATALOG}"
    )
else:
    print("  nothing to clean up")

print("\n✅ Ready for fresh run")

Cleaning up leftover resources...
  nothing to clean up

✅ Ready for fresh run


In [3]:
# Force-clean stuck entities in Polaris async cleanup/trash namespaces
# Polaris moves failed-purge entities to namespaces like 'polaris-cleanup-trash-ns'.
# These must be purged with drop-with-purge enabled.
print("Checking for stuck entities in trash namespaces...")

# Ensure purge enabled on catalog (idempotent)
catalogs = requests.get(f"{BASE_MGMT}/catalogs", headers=h(tok)).json().get("catalogs", [])
if any(c["name"] == PRACTICE_CATALOG for c in catalogs):
    rg = requests.get(f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}", headers=h(tok))
    cat = rg.json()
    ver = cat["entityVersion"]
    merged = {**cat.get("properties", {}),
              "polaris.config.drop-with-purge.enabled": "true",
              "polaris.config.purge-view-metadata-on-drop": "false"}
    requests.put(f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}", headers=h(tok), json={
        "currentEntityVersion": ver, "properties": merged
    })
    requests.put(f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}/catalog-roles/catalog_admin/grants",
                 headers=h(tok),
                 json={"grant": {"type": "catalog", "privilege": "CATALOG_MANAGE_CONTENT"}})
    print("  granted CATALOG_MANAGE_CONTENT for purge")


    # List ALL namespaces including trash ones
    ns_r = requests.get(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces", headers=h(tok))
    all_ns = ns_r.json().get("namespaces", [])
    print(f"  namespaces found: {all_ns}")

    for ns in all_ns:
        ns_name = ns[0] if isinstance(ns, list) else ns

        # Force-delete all tables with purge
        t_r = requests.get(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{ns_name}/tables", headers=h(tok))
        for t in t_r.json().get("identifiers", []):
            r = requests.delete(
                f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{ns_name}/tables/{t['name']}?purgeRequested=true",
                headers=h(tok))
            print(f"  table {ns_name}.{t['name']}: {r.status_code}")

        # Force-delete all views with purge
        v_r = requests.get(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{ns_name}/views", headers=h(tok))
        for v in v_r.json().get("identifiers", []):
            r = requests.delete(
                f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{ns_name}/views/{v['name']}?purgeRequested=true",
                headers=h(tok))
            print(f"  view {ns_name}.{v['name']}: {r.status_code}")
            if r.status_code not in (200, 204, 404):
                print(f"     ⚠️  {r.text[:200]}")

        # Delete the namespace
        r = requests.delete(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{ns_name}", headers=h(tok))
        print(f"  namespace {ns_name}: {r.status_code}")

    print("✅ Stuck entity sweep complete")
else:
    print("  catalog doesn't exist — nothing stuck")

Checking for stuck entities in trash namespaces...
  catalog doesn't exist — nothing stuck


## Part 1 — Create catalog, namespace, table, view

In [4]:
# Step 1: Create practice catalog
# Key config (from Polaris 1.3.0 docs):
#   PURGE_VIEW_METADATA_ON_DROP default=true → view drop tries to delete MinIO files
#     → causes 403 if storage creds unavailable. We disable it for clean view drops.
#   DROP_WITH_PURGE_ENABLED default=false → table purge blocked until enabled
print("Step 1: Create practice catalog")
r = requests.post(f"{BASE_MGMT}/catalogs", headers=h(tok), json={
    "catalog": {
        "name": PRACTICE_CATALOG,
        "type": "INTERNAL",
        "properties": {
            "default-base-location": PRACTICE_LOCATION,
            "polaris.config.drop-with-purge.enabled": "false",
            "polaris.config.purge-view-metadata-on-drop": "false",
        },
        "storageConfigInfo": {
            "storageType": "S3",
            "allowedLocations": [PRACTICE_LOCATION],
            "pathStyleAccess": True,
            "endpoint": "http://192.168.139.2:9000",
            "endpointInternal": "http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000",
        },
    }
})
print_response(r, "Create catalog")
catalog_version = r.json().get("entityVersion", 1)
print(f"\n✅ entityVersion: {catalog_version}")


Step 1: Create practice catalog

Create catalog:
  Status:     201
  Request-Id: d2499e4f-0c44-4779-9aa0-860d9d3a8f33_0000000000000021738
  Body: {
  "type": "INTERNAL",
  "name": "purge-practice",
  "properties": {
    "polaris.config.drop-with-purge.enabled": "false",
    "polaris.config.purge-view-metadata-on-drop": "false",
    "default-base-location": "s3a://data-catalog-bucket/purge-practice/"
  },
  "createTimestamp": 1782803223209,
  "lastUpdateTimestamp": 0,
  "entityVersion": 1,
  "storageConfigInfo": {
    "endpoint": "http://192.168.139.2:9000",
    "endpointInternal": "http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000",
    "pathStyleAccess": true,
    "storageType": "S3",
    "allowedLocations": [
      "s3a://data-catalog-bucket/purge-practice/"
    ]
  }
}

✅ entityVersion: 1


In [5]:
# Step 1b: Grant CATALOG_MANAGE_CONTENT to catalog_admin role
# Required for DROP_TABLE_WITH_PURGE — purge deletes data files, which needs
# TABLE_WRITE_DATA. Plain catalog_admin lacks it; CATALOG_MANAGE_CONTENT bundles it.
print("Step 1b: Grant purge privilege to catalog_admin")
r = requests.put(
    f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}/catalog-roles/catalog_admin/grants",
    headers=h(tok),
    json={"grant": {"type": "catalog", "privilege": "CATALOG_MANAGE_CONTENT"}}
)
print(f"  grant CATALOG_MANAGE_CONTENT: {r.status_code}")
if r.status_code not in (200, 201):
    print(f"  {r.text[:200]}")
else:
    print("  ✅ catalog_admin can now DROP_TABLE_WITH_PURGE")


Step 1b: Grant purge privilege to catalog_admin
  grant CATALOG_MANAGE_CONTENT: 201
  ✅ catalog_admin can now DROP_TABLE_WITH_PURGE


In [6]:
# Step 2: Create namespace
print("Step 2: Create namespace")
r = create_namespace(catalog=PRACTICE_CATALOG, ns=PRACTICE_NS, token=tok)
print_response(r, "Create namespace")


Step 2: Create namespace

Create namespace:
  Status:     500
  Request-Id: d2499e4f-0c44-4779-9aa0-860d9d3a8f33_0000000000000021740
  Body: {
  "error": {
    "message": "Cannot invoke \"org.apache.polaris.core.persistence.PolarisResolvedPathWrapper.getRawLeafEntity()\" because the return value of \"org.apache.polaris.core.persistence.resolver.PolarisResolutionManifest.getPassthroughResolvedPath(Object)\" is null",
    "type": "NullPointerException",
    "code": 500
  }
}


In [7]:
# Step 3: Create table
print("Step 3: Create table")
r = requests.post(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables",
    headers=h(tok),
    json={
        "name": PRACTICE_TABLE,
        "location": f"{PRACTICE_LOCATION}{PRACTICE_NS}/{PRACTICE_TABLE}/",
        "schema": {
            "type": "struct",
            "fields": [
                {"id": 1, "name": "id",     "type": "long",   "required": True},
                {"id": 2, "name": "name",   "type": "string", "required": False},
                {"id": 3, "name": "amount", "type": "double", "required": False},
            ]
        },
        "properties": {"write.format.default": "parquet"}
    }
)
print_response(r, "Create table")
# Save current snapshot for commit requirement
table_meta      = r.json().get("metadata", {})
current_snapshot = table_meta.get("current-snapshot-id", -1)
table_location   = table_meta.get("location", f"{PRACTICE_LOCATION}{PRACTICE_NS}/{PRACTICE_TABLE}")
print(f"\n✅ Table created  current-snapshot: {current_snapshot}")


Step 3: Create table

Create table:
  Status:     200
  Request-Id: d2499e4f-0c44-4779-9aa0-860d9d3a8f33_0000000000000021741
  Body: {
  "metadata-location": "s3a://data-catalog-bucket/purge-practice/practice-ns/practice-table/metadata/00000-0e70a578-fd59-4996-a723-2ce644a5e2d0.metadata.json",
  "metadata": {
    "format-version": 2,
    "table-uuid": "9cf251b5-9370-41ad-80a3-20e1df030e8e",
    "location": "s3a://data-catalog-bucket/purge-practice/practice-ns/practice-table",
    "last-sequence-number": 0,
    "last-updated-ms": 1782803233201,
    "last-column-id": 3,
    "current-schema-id": 0,
    "schemas": [
      {
        "type": "struct",
        "schema-id": 0,
        "fields": [
          {
            "id": 1,
            "name": "id",
            "required": true,
            "type": "long"
          },
          {
            "id": 2,
            "name": "name",
            "required": false,
            "type": "string"
          },
          {
            "id": 3,
      

In [8]:
# Step 4: Create view
print("Step 4: Create view")
r = requests.post(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/views",
    headers=h(tok),
    json={
        "name": PRACTICE_VIEW,
        "default-namespace": [PRACTICE_NS],
        "schema": {
            "type": "struct",
            "schema-id": 0,
            "fields": [
                {"id": 1, "name": "id",   "type": "long",   "required": True},
                {"id": 2, "name": "name", "type": "string", "required": False},
            ]
        },
        "view-version": {
            "version-id": 1,
            "timestamp-ms": int(__import__("time").time() * 1000),
            "schema-id": 0,
            "default-namespace": [PRACTICE_NS],
            "summary": {"engine": "spark"},
            "representations": [{
                "type": "sql",
                "sql": f"SELECT id, name FROM {PRACTICE_TABLE}",
                "dialect": "spark"
            }]
        },
        "location": f"{PRACTICE_LOCATION}{PRACTICE_NS}/{PRACTICE_VIEW}/",
        "properties": {}
    }
)
print_response(r, "Create view")


Step 4: Create view

Create view:
  Status:     500
  Request-Id: d2499e4f-0c44-4779-9aa0-860d9d3a8f33_0000000000000021742
  Body: {
  "error": {
    "message": "metadata",
    "type": "NullPointerException",
    "code": 500
  }
}


## Part 1b — Write data to table via Iceberg REST

```
Iceberg v2 write flow (without Spark):
  Step A: write Parquet file to MinIO (pyarrow + s3fs)
  Step B: write manifest + manifest-list Avro files to MinIO
  Step C: commit snapshot via updateTable API

Polaris 1.3.0 commit rules (confirmed from testing):
  → requirements can be []
  → add-snapshot only (no set-snapshot-ref in same request)
  → no sequence-number field (Polaris manages internally)
  → manifest-list must be real path (not empty string)
```


In [9]:
# Step A: Write Parquet file to MinIO
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs, uuid, io, time
try:
    import fastavro
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "fastavro", "--quiet"])
    import fastavro

print("Step A: Write Parquet file to MinIO")

table_data = pa.table({
    "id":     pa.array([1, 2, 3, 4, 5], type=pa.int64()),
    "name":   pa.array(["alice", "bob", "carol", "dave", "eve"], type=pa.string()),
    "amount": pa.array([100.0, 200.5, 300.0, 150.75, 250.0], type=pa.float64()),
})

file_uuid    = uuid.uuid4().hex
data_dir     = f"{PRACTICE_LOCATION}{PRACTICE_NS}/{PRACTICE_TABLE}/data"
parquet_path = f"{data_dir}/{file_uuid}.parquet"
s3_path      = parquet_path.replace("s3a://", "s3://")

fs = s3fs.S3FileSystem(
    key=MINIO_ACCESS_KEY,
    secret=MINIO_SECRET_KEY,
    use_ssl=False,
    client_kwargs={"endpoint_url": MINIO_ENDPOINT}
)

with fs.open(s3_path, "wb") as f_out:
    pq.write_table(table_data, f_out)

FILE_SIZE_BYTES = fs.info(s3_path)["size"]
print(f"  ✅ Parquet written: {FILE_SIZE_BYTES} bytes, {len(table_data)} rows")
print(f"  path: {parquet_path}")
print()
print("📸 MinIO: data file visible now (no snapshot yet)")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")


Step A: Write Parquet file to MinIO
  ✅ Parquet written: 1097 bytes, 5 rows
  path: s3a://data-catalog-bucket/purge-practice/practice-ns/practice-table/data/c62d3617bcbb4e82a8c2e66c9103bc7f.parquet

📸 MinIO: data file visible now (no snapshot yet)
   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F


In [10]:
# Step B: Write Iceberg v2 manifest + manifest-list to MinIO
print("Step B: Write manifest + manifest-list")

snapshot_id = int(time.time() * 1000)
commit_ts   = snapshot_id
seq_num     = 1
meta_dir    = f"{PRACTICE_LOCATION}{PRACTICE_NS}/{PRACTICE_TABLE}/metadata"

MANIFEST_SCHEMA = {
    "type": "record", "name": "manifest_entry",
    "fields": [
        {"name": "status",          "type": "int"},
        {"name": "snapshot_id",     "type": ["null","long"], "default": None},
        {"name": "sequence_number", "type": ["null","long"], "default": None},
        {"name": "file_sequence_number","type":["null","long"],"default":None},
        {"name": "data_file", "type": {
            "type": "record", "name": "r2",
            "fields": [
                {"name": "content",            "type": "int"},
                {"name": "file_path",          "type": "string"},
                {"name": "file_format",        "type": "string"},
                {"name": "partition",          "type": {"type":"record","name":"r102","fields":[]}},
                {"name": "record_count",       "type": "long"},
                {"name": "file_size_in_bytes", "type": "long"},
                {"name": "column_sizes",       "type":["null",{"type":"array","items":{"type":"record","name":"k81v81","fields":[{"name":"key","type":"int"},{"name":"value","type":"long"}]}}],"default":None},
                {"name": "value_counts",       "type":["null",{"type":"array","items":"k81v81"}],"default":None},
                {"name": "null_value_counts",  "type":["null",{"type":"array","items":"k81v81"}],"default":None},
                {"name": "nan_value_counts",   "type":["null",{"type":"array","items":"k81v81"}],"default":None},
                {"name": "lower_bounds",       "type":["null",{"type":"array","items":{"type":"record","name":"k81v100","fields":[{"name":"key","type":"int"},{"name":"value","type":"bytes"}]}}],"default":None},
                {"name": "upper_bounds",       "type":["null",{"type":"array","items":"k81v100"}],"default":None},
                {"name": "key_metadata",       "type":["null","bytes"],"default":None},
                {"name": "split_offsets",      "type":["null",{"type":"array","items":"long"}],"default":None},
                {"name": "equality_ids",       "type":["null",{"type":"array","items":"int"}],"default":None},
                {"name": "sort_order_id",      "type":["null","int"],"default":None},
            ]
        }}
    ]
}

manifest_entry = {
    "status": 1, "snapshot_id": snapshot_id,
    "sequence_number": seq_num, "file_sequence_number": seq_num,
    "data_file": {
        "content": 0, "file_path": parquet_path, "file_format": "PARQUET",
        "partition": {}, "record_count": 5, "file_size_in_bytes": FILE_SIZE_BYTES,
        "column_sizes": None, "value_counts": None, "null_value_counts": None,
        "nan_value_counts": None, "lower_bounds": None, "upper_bounds": None,
        "key_metadata": None, "split_offsets": None, "equality_ids": None, "sort_order_id": 0,
    }
}

manifest_path   = f"{meta_dir}/{uuid.uuid4().hex}-m0.avro"
s3_manifest     = manifest_path.replace("s3a://", "s3://")
buf = io.BytesIO()
fastavro.writer(buf, fastavro.parse_schema(MANIFEST_SCHEMA), [manifest_entry],
    metadata={
        "schema": '{"type":"struct","schema-id":0,"fields":[{"id":1,"name":"id","required":true,"type":"long"},{"id":2,"name":"name","required":false,"type":"string"},{"id":3,"name":"amount","required":false,"type":"double"}]}',
        "partition-spec": '[]', "partition-spec-id": '0', "format-version": '2',
        "content": 'data', "sequence-number": str(seq_num),
        "min-sequence-number": str(seq_num), "snapshot-id": str(snapshot_id),
        "added-data-files": '1', "existing-data-files": '0', "deleted-data-files": '0',
        "added-rows": '5', "existing-rows": '0', "deleted-rows": '0',
    })
manifest_bytes = buf.getvalue()
manifest_size  = len(manifest_bytes)
with fs.open(s3_manifest, "wb") as f: f.write(manifest_bytes)
print(f"  ✅ manifest: {s3_manifest} ({manifest_size}B)")

MANIFEST_LIST_SCHEMA = {
    "type": "record", "name": "manifest_file",
    "fields": [
        {"name": "manifest_path",              "type": "string"},
        {"name": "manifest_length",             "type": "long"},
        {"name": "partition_spec_id",           "type": "int"},
        {"name": "content",                     "type": "int",  "default": 0},
        {"name": "sequence_number",             "type": "long", "default": 0},
        {"name": "min_sequence_number",         "type": "long", "default": 0},
        {"name": "added_snapshot_id",           "type": "long"},
        {"name": "added_data_files_count",      "type": "int"},
        {"name": "existing_data_files_count",   "type": "int"},
        {"name": "deleted_data_files_count",    "type": "int"},
        {"name": "added_rows_count",            "type": "long"},
        {"name": "existing_rows_count",         "type": "long"},
        {"name": "deleted_rows_count",          "type": "long"},
        {"name": "partitions",  "type":["null",{"type":"array","items":{"type":"record","name":"r508","fields":[{"name":"contains_null","type":"boolean"},{"name":"contains_nan","type":["null","boolean"],"default":None},{"name":"lower_bound","type":["null","bytes"],"default":None},{"name":"upper_bound","type":["null","bytes"],"default":None}]}}],"default":None},
        {"name": "key_metadata","type":["null","bytes"],"default":None},
    ]
}

manifest_list_path = f"{meta_dir}/snap-{snapshot_id}-{uuid.uuid4().hex}.avro"
s3_ml = manifest_list_path.replace("s3a://", "s3://")
buf2 = io.BytesIO()
fastavro.writer(buf2, fastavro.parse_schema(MANIFEST_LIST_SCHEMA), [{
    "manifest_path": manifest_path, "manifest_length": manifest_size,
    "partition_spec_id": 0, "content": 0, "sequence_number": seq_num,
    "min_sequence_number": seq_num, "added_snapshot_id": snapshot_id,
    "added_data_files_count": 1, "existing_data_files_count": 0, "deleted_data_files_count": 0,
    "added_rows_count": 5, "existing_rows_count": 0, "deleted_rows_count": 0,
    "partitions": None, "key_metadata": None,
}])
with fs.open(s3_ml, "wb") as f: f.write(buf2.getvalue())
print(f"  ✅ manifest-list: {s3_ml}")
print()
print("📸 MinIO: metadata/ should now have .avro files")


Step B: Write manifest + manifest-list
  ✅ manifest: s3://data-catalog-bucket/purge-practice/practice-ns/practice-table/metadata/663ee6bdb9f14ee79047986aa51df4f2-m0.avro (2506B)
  ✅ manifest-list: s3://data-catalog-bucket/purge-practice/practice-ns/practice-table/metadata/snap-1782803242189-0ac83647f63f415ab3c88506d031ab1f.avro

📸 MinIO: metadata/ should now have .avro files


In [11]:
# Step C: Commit snapshot (Polaris 1.3.0 confirmed working body)
print("Step C: Commit snapshot")

r_commit = requests.post(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}",
    headers=h(tok),
    json={
        "identifier": {"namespace": [PRACTICE_NS], "name": PRACTICE_TABLE},
        "requirements": [],
        "updates": [{
            "action": "add-snapshot",
            "snapshot": {
                "snapshot-id":   snapshot_id,
                "timestamp-ms":  commit_ts,
                "summary": {
                    "operation":         "append",
                    "added-data-files":  "1",
                    "added-records":     "5",
                    "added-files-size":  str(FILE_SIZE_BYTES),
                    "total-data-files":  "1",
                    "total-records":     "5",
                },
                "schema-id":     0,
                "manifest-list": manifest_list_path,
            }
        }]
    }
)
print_response(r_commit, "Commit snapshot")

if r_commit.status_code in (200, 201, 204):
    # Set main branch ref (separate request)
    r_ref = requests.post(
        f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}",
        headers=h(tok),
        json={
            "identifier": {"namespace": [PRACTICE_NS], "name": PRACTICE_TABLE},
            "requirements": [],
            "updates": [{
                "action": "set-snapshot-ref",
                "ref-name": "main",
                "type": "branch",
                "snapshot-id": snapshot_id,
            }]
        }
    )
    current = r_ref.json().get("metadata", {}).get("current-snapshot-id", -1)
    print(f"\n  set-snapshot-ref: HTTP {r_ref.status_code}")
    print(f"  current-snapshot: {current}")
    print(f"\n✅ 5 rows committed — visible to Spark/Trino via Polaris")


Step C: Commit snapshot

Commit snapshot:
  Status:     200
  Request-Id: d2499e4f-0c44-4779-9aa0-860d9d3a8f33_0000000000000021743
  Body: {
  "metadata-location": "s3a://data-catalog-bucket/purge-practice/practice-ns/practice-table/metadata/00000-0e70a578-fd59-4996-a723-2ce644a5e2d0.metadata.json",
  "metadata": {
    "format-version": 2,
    "table-uuid": "9cf251b5-9370-41ad-80a3-20e1df030e8e",
    "location": "s3a://data-catalog-bucket/purge-practice/practice-ns/practice-table",
    "last-sequence-number": 0,
    "last-updated-ms": 1782803233201,
    "last-column-id": 3,
    "current-schema-id": 0,
    "schemas": [
      {
        "type": "struct",
        "schema-id": 0,
        "fields": [
          {
            "id": 1,
            "name": "id",
            "required": true,
            "type": "long"
          },
          {
            "id": 2,
            "name": "name",
            "required": false,
            "type": "string"
          },
          {
            "id": 3,


In [12]:
# Step D: Verify snapshot
print("Step D: Verify")
r = requests.get(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}",
    headers=h(tok)
)
meta      = r.json().get("metadata", {})
snapshots = meta.get("snapshots", [])
current   = meta.get("current-snapshot-id", -1)
refs      = meta.get("refs", {})

print(f"  snapshots:        {len(snapshots)}")
print(f"  current-snapshot: {current}")
print(f"  refs:             {refs}")
for s in snapshots:
    sm = s.get("summary", {})
    print(f"  → id={s.get('snapshot-id')} op={sm.get('operation')} records={sm.get('added-records','?')}")

print()
if current == snapshot_id:
    print("✅ Snapshot confirmed — table has 5 rows")
    print()
    print("📸 MinIO final state before purge:")
    print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")
    print("   data/     → .parquet (5 rows)")
    print("   metadata/ → -m0.avro (manifest)")
    print("              snap-*.avro (manifest-list)")
    print("              .metadata.json (table metadata)")
else:
    print(f"⚠️  current-snapshot={current} ≠ {snapshot_id}")


Step D: Verify
  snapshots:        1
  current-snapshot: 1782803242189
  refs:             {'main': {'snapshot-id': 1782803242189, 'type': 'branch'}}
  → id=1782803242189 op=append records=5

✅ Snapshot confirmed — table has 5 rows

📸 MinIO final state before purge:
   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F
   data/     → .parquet (5 rows)
   metadata/ → -m0.avro (manifest)
              snap-*.avro (manifest-list)
              .metadata.json (table metadata)


## Part 2 — Observe MinIO before purge

In [13]:
# Verify all entities exist
print("Verify entities exist before purge:")
r_t = requests.get(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}", headers=h(tok))
r_v = requests.get(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/views/{PRACTICE_VIEW}", headers=h(tok))
print(f"  Table {PRACTICE_TABLE}: HTTP {r_t.status_code} {'✅' if r_t.status_code == 200 else '❌'}")
print(f"  View  {PRACTICE_VIEW}:  HTTP {r_v.status_code} {'✅' if r_v.status_code == 200 else '❌'}")
print()
print("📸 Check MinIO NOW (before any purge)")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")


Verify entities exist before purge:
  Table practice-table: HTTP 200 ✅
  View  practice-view:  HTTP 200 ✅

📸 Check MinIO NOW (before any purge)
   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F


## Part 3 — View purge

```
Two separate configs control view drop behavior (Polaris 1.3.0):

PURGE_VIEW_METADATA_ON_DROP (default: true)
  → when true, dropping a view ALWAYS deletes its metadata file from MinIO
  → needs storage credentials → 403 if creds unavailable
  → we set this FALSE on the catalog → view drop is metadata-only (PostgreSQL)

DROP_WITH_PURGE_ENABLED (default: false)
  → controls whether purgeRequested=true is allowed (mainly for TABLES)
  → views have a tiny metadata file but no Parquet data

For this practice:
  → catalog created with purge-view-metadata-on-drop = false
  → view drop removes PostgreSQL metadata, leaves any MinIO file untouched
  → demonstrates views have negligible storage footprint
```


In [14]:
# Drop VIEW — metadata-only (purge-view-metadata-on-drop=false on catalog)
print("Drop view")
r = requests.delete(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/views/{PRACTICE_VIEW}",
    headers=h(tok)
)
print_response(r, "Drop view")
print()
print("📸 Check MinIO AFTER view drop")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")
print("Expected: table data still present, view had no Parquet data")


Drop view

Drop view:
  Status:     204
  Request-Id: d2499e4f-0c44-4779-9aa0-860d9d3a8f33_0000000000000021748
  Body: 

📸 Check MinIO AFTER view drop
   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F
Expected: table data still present, view had no Parquet data


## Part 4 — Table purge (Parquet files removed from MinIO)

In [15]:
# Enable DROP_WITH_PURGE on catalog (required for TABLE purge)
# Table purge deletes Parquet + metadata files from MinIO → needs this flag
print("Enable DROP_WITH_PURGE on catalog")
r_get = requests.get(f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}", headers=h(tok))
cat = r_get.json()
version = cat["entityVersion"]
merged = {**cat.get("properties", {}), "polaris.config.drop-with-purge.enabled": "true"}
r_patch = requests.put(
    f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}",
    headers=h(tok),
    json={"currentEntityVersion": version, "properties": merged}
)
print_response(r_patch, "Enable purge")

Enable DROP_WITH_PURGE on catalog

Enable purge:
  Status:     200
  Request-Id: d2499e4f-0c44-4779-9aa0-860d9d3a8f33_0000000000000021750
  Body: {
  "type": "INTERNAL",
  "name": "purge-practice",
  "properties": {
    "polaris.config.drop-with-purge.enabled": "true",
    "polaris.config.purge-view-metadata-on-drop": "false",
    "default-base-location": "s3a://data-catalog-bucket/purge-practice/"
  },
  "createTimestamp": 1782803223209,
  "lastUpdateTimestamp": 1782803251030,
  "entityVersion": 2,
  "storageConfigInfo": {
    "endpoint": "http://192.168.139.2:9000",
    "endpointInternal": "http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000",
    "pathStyleAccess": true,
    "storageType": "S3",
    "allowedLocations": [
      "s3a://data-catalog-bucket/purge-practice/"
    ]
  }
}


In [16]:
# Drop TABLE with purge
print("Drop table WITH purgeRequested=true")
print()
print("📸 MinIO BEFORE table purge — note files present")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")
print()

r = requests.delete(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}?purgeRequested=true",
    headers=h(tok)
)
print_response(r, "Drop table (purge)")
print()
print("📸 MinIO AFTER table purge")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")
print("Expected: data/ + metadata/ GONE")


Drop table WITH purgeRequested=true

📸 MinIO BEFORE table purge — note files present
   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F


Drop table (purge):
  Status:     204
  Request-Id: d2499e4f-0c44-4779-9aa0-860d9d3a8f33_0000000000000021751
  Body: 

📸 MinIO AFTER table purge
   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F
Expected: data/ + metadata/ GONE


In [17]:
# Verify table gone
r = requests.get(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}", headers=h(tok))
print(f"Table after purge: HTTP {r.status_code} {'✅ 404=gone' if r.status_code == 404 else '❌ still exists'}")


Table after purge: HTTP 404 ✅ 404=gone


## Part 5 — Catalog purge (base location removed from MinIO)

In [18]:
# Drop namespace first (required before catalog)
print("Drop namespace")
r = requests.delete(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}", headers=h(tok))
print_response(r, "Drop namespace")


Drop namespace

Drop namespace:
  Status:     204
  Request-Id: d2499e4f-0c44-4779-9aa0-860d9d3a8f33_0000000000000021753
  Body: 


In [19]:
# Drop CATALOG with purge
print("Drop catalog WITH purgeRequested=true")
print()
print("📸 MinIO BEFORE catalog purge")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")
print()

r = requests.delete(
    f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}?purgeRequested=true",
    headers=h(tok)
)
print_response(r, "Drop catalog (purge)")
print()
print("📸 MinIO AFTER catalog purge")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")
print("Expected: purge-practice/ prefix GONE")


Drop catalog WITH purgeRequested=true

📸 MinIO BEFORE catalog purge
   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F


Drop catalog (purge):
  Status:     204
  Request-Id: d2499e4f-0c44-4779-9aa0-860d9d3a8f33_0000000000000021754
  Body: 

📸 MinIO AFTER catalog purge
   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F
Expected: purge-practice/ prefix GONE


## Part 6 — Verification

In [21]:
print("Final verification")
r = requests.get(f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}", headers=h(tok))
print(f"Catalog: HTTP {r.status_code} {'✅ 404=gone' if r.status_code == 404 else '❌ still exists'}")

all_cats = [c["name"] for c in requests.get(f"{BASE_MGMT}/catalogs", headers=h(tok)).json().get("catalogs", [])]
print(f"All catalogs: {all_cats}")
print(f"purge-practice present: {'❌' if PRACTICE_CATALOG in all_cats else '✅ removed'}")


Final verification
Catalog: HTTP 404 ✅ 404=gone
All catalogs: ['data-lifecycle-practice', 'privmx-viewcreate-viewlist-54d6', 'privmx-viewget-tablelist-7b34', 'vptest-c1', 'vptest-c8']
purge-practice present: ✅ removed


## Summary — What we learned

```
Polaris 1.3.0 purge config (from official docs):

PURGE_VIEW_METADATA_ON_DROP (default: true)
  → view drop deletes metadata file from MinIO automatically
  → needs storage credentials → 403 if unavailable
  → set false → view drop is PostgreSQL-only (what we used)

DROP_WITH_PURGE_ENABLED (default: false)
  → must be true to allow purgeRequested=true on tables
  → table purge deletes Parquet + metadata from MinIO

CLEANUP_ON_NAMESPACE_DROP (default: false)
CLEANUP_ON_CATALOG_DROP (default: false)
  → deprecated flags; control data cleanup on namespace/catalog drop

Behavior observed:
  View  drop  → PostgreSQL metadata removed; MinIO untouched (config off)
  Table purge → PostgreSQL + MinIO Parquet/metadata removed
  Catalog purge → all metadata + base location removed

Order (always bottom-up):
  table → namespace → catalog

Production rule:
  bronze/silver/gold:
    → DROP_WITH_PURGE_ENABLED = false (protect data)
    → PURGE_VIEW_METADATA_ON_DROP = true is fine (views regenerable)
  test/watchdog:
    → DROP_WITH_PURGE_ENABLED = true (clean test data)

Iceberg commit quirks (Polaris 1.3.0):
  → no sequence-number field in snapshot
  → add-snapshot + set-snapshot-ref must be separate requests
  → manifest-list must be a real written path
```
